# Revision visual con MediaPipe

Objetivo: detectar clips que no sirven para VSR porque no hay una boca clara de una sola persona enfocada.

Este notebook no borra nada. Primero calcula metricas visuales sobre una muestra de clips y despues muestra video + texto + razones.

In [1]:
from pathlib import Path
from IPython.display import HTML, Video, display
import pandas as pd

def encontrar_raiz(path=None):
    path = Path.cwd() if path is None else Path(path)
    for candidato in [path, *path.parents]:
        if (candidato / 'data' / 'clips').exists():
            return candidato
    raise RuntimeError('No encontre la raiz del repo')

RAIZ = encontrar_raiz()
CLIPS_DIR = RAIZ / 'data' / 'clips'
print(RAIZ)

C:\Users\bianc\NLP Natural Language Processing\labios-argentos


## Dependencias

Si esta celda dice que falta `mediapipe`, instalar en un entorno de preprocesamiento:

```bash
pip install -r visual_preprocessing/requirements.txt
```

In [2]:
import sys
sys.path.insert(0, str(RAIZ))

try:
    import mediapipe as mp
    MEDIAPIPE_OK = True
    print('mediapipe OK', mp.__version__)
except Exception as e:
    MEDIAPIPE_OK = False
    print('Falta mediapipe:', e)

Falta mediapipe: No module named 'mediapipe'


## Armar tabla de clips

Elegimos una fuente y una cantidad chica al principio. Despues subimos `N` o cambiamos `FUENTE`.

In [3]:
FUENTE = None  # ejemplo: 'JULI POGGIO EN FERNÉ CON GREGO'
N = 30

filas = []
fuentes = [CLIPS_DIR / FUENTE] if FUENTE else sorted(p for p in CLIPS_DIR.iterdir() if p.is_dir())
for fuente in fuentes:
    for clip_path in sorted(fuente.glob('*.mp4')):
        text_path = clip_path.with_suffix('.txt')
        filas.append({
            'source_title': fuente.name,
            'clip_id': clip_path.stem,
            'clip_path': clip_path,
            'text_path': text_path,
            'text': text_path.read_text(encoding='utf-8').strip() if text_path.exists() else '',
        })

clips_df = pd.DataFrame(filas)
clips_df.head(), len(clips_df)

(                                 source_title    clip_id  \
 0  ANCDOTA VIAJE MUNDIAL BRASIL 2014  parte 1  clip_0000   
 1  ANCDOTA VIAJE MUNDIAL BRASIL 2014  parte 1  clip_0001   
 2  ANCDOTA VIAJE MUNDIAL BRASIL 2014  parte 1  clip_0002   
 3  ANCDOTA VIAJE MUNDIAL BRASIL 2014  parte 1  clip_0003   
 4  ANCDOTA VIAJE MUNDIAL BRASIL 2014  parte 1  clip_0004   
 
                                            clip_path  \
 0  C:\Users\bianc\NLP Natural Language Processing...   
 1  C:\Users\bianc\NLP Natural Language Processing...   
 2  C:\Users\bianc\NLP Natural Language Processing...   
 3  C:\Users\bianc\NLP Natural Language Processing...   
 4  C:\Users\bianc\NLP Natural Language Processing...   
 
                                            text_path  \
 0  C:\Users\bianc\NLP Natural Language Processing...   
 1  C:\Users\bianc\NLP Natural Language Processing...   
 2  C:\Users\bianc\NLP Natural Language Processing...   
 3  C:\Users\bianc\NLP Natural Language Processing...   
 4 

## Auditar con MediaPipe

Esto muestrea algunos frames por clip. Detecta casos con 0 caras, multiples caras, boca poco visible o boca descentrada.

In [4]:
if MEDIAPIPE_OK:
    from visual_preprocessing.src.auditar_calidad_visual import analizar_clip, crear_landmarker

    landmarker, mp_runtime = crear_landmarker(num_faces=3)
    resultados = []
    try:
        for _, row in clips_df.head(N).iterrows():
            met = analizar_clip(row.clip_path, texto=row.text, n_frames=7, landmarker=landmarker, mp=mp_runtime)
            resultados.append({**row.to_dict(), **met})
    finally:
        landmarker.close()

    visual_df = pd.DataFrame(resultados)
else:
    visual_df = pd.DataFrame()
    print('Instala mediapipe para correr esta auditoria visual.')

visual_df

Instala mediapipe para correr esta auditoria visual.


""


## Ver clips marcados para revision

In [5]:
if not visual_df.empty:
    display(visual_df['visual_status'].value_counts())
    display(visual_df[['source_title', 'clip_id', 'max_caras', 'ratio_cara', 'ratio_multi_cara', 'ratio_boca_visible', 'visual_status', 'visual_reasons']])

def ver_clip(i, tabla=None, ancho=520):
    if tabla is None:
        tabla = visual_df[visual_df.visual_status == 'review'].reset_index(drop=True)
    row = tabla.iloc[i]
    display(HTML(f'<h3>{i} - {row.source_title} / {row.clip_id}</h3>'))
    display(HTML(f'<p><b>razones:</b> {row.visual_reasons}</p>'))
    display(HTML(f'<p><b>caras max:</b> {row.max_caras} | <b>ratio cara:</b> {row.ratio_cara} | <b>ratio multi:</b> {row.ratio_multi_cara} | <b>ratio boca visible:</b> {row.ratio_boca_visible}</p>'))
    display(Video(str(row.clip_path), embed=True, width=ancho))
    print(row.text)

if not visual_df.empty and (visual_df.visual_status == 'review').any():
    ver_clip(0)